# **Multi-Tenancy RAG System with LlamaIndex — Production Version**

This extends the original demo notebook into something closer to
production-ready, while staying a single notebook:

- **Structural isolation** — there is no code path that can build a query
  engine, ingest, or delete without a `tenant_id`. Every ingested node is
  verified to be correctly tagged before it's written to the index.
- **Persistent storage** — backed by **Qdrant** (on-disk by default, or a
  real Qdrant server/cloud instance by setting one env var) instead of an
  in-memory index that vanishes when the runtime resets.
- **Local Hugging Face embeddings** — dense embeddings run locally via
  `BAAI/bge-small-en-v1.5` (sentence-transformers) instead of calling the
  Gemini embedding API, so ingestion isn't subject to embedding API rate
  limits. The LLM still uses Gemini.
- **3+ tenants** — generalized beyond two hardcoded users.
- **Hybrid search** — dense (embeddings) + sparse (BM25-style) retrieval.
- **Tenant deletion** — purge a tenant's data on request.
- **Real retry/backoff** — exponential backoff on rate-limit errors,
  replacing a fixed `time.sleep(100)` guess.
- **Inline isolation tests** — a verification cell that asserts no tenant
  can ever see another tenant's content, so isolation is checked, not just
  assumed.

1. Setup
2. Configure Gemini API key
3. Imports & configuration
4. Configure LLM + embedding models
5. Persistent multi-tenant vector store (Qdrant)
6. Download data
7. Load data
8. Ingestion pipeline (with retry)
9. Hardened ingestion function
10. Ingest documents for each tenant
11. Hardened query engine factory
12. Querying
13. Isolation verification (inline tests)
14. Tenant deletion / lifecycle
15. Inspecting stored chunks in Qdrant
16. Production notes & caveats

## 1. Setup

Installs `llama-index`, the Gemini LLM/embedding integrations, the Qdrant
vector store integration, `fastembed` (for hybrid/sparse search), and
`tenacity` (for retry/backoff).

**Dependency note:** as of this writing, `llama-index-vector-stores-qdrant`
declares it needs `qdrant-client>=1.16.0`, but that combination currently
throws `ImportError: cannot import name 'IDF_EMBEDDING_MODELS'` at import
time. We pin `qdrant-client==1.12.2` below, which is compatible. Re-check
upstream before bumping either package.

In [1]:
!pip install -q llama-index pypdf llama-index-llms-google-genai llama-index-embeddings-huggingface \
    llama-index-readers-file llama-index-vector-stores-qdrant "qdrant-client==1.12.2" fastembed tenacity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.2/267.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/32

## 2. Configure Gemini API key

In [2]:
from google import genai


from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")


client = genai.Client(api_key=api_key)

## 3. Imports & configuration

`TENANT_KEY` is defined once and reused everywhere (ingestion, querying,
deletion) so isolation logic can't drift out of sync between different
parts of the notebook.

In [3]:
import logging
import time
from pathlib import Path

from llama_index.core import (
    Settings,
    StorageContext,
    VectorStoreIndex,
    SimpleDirectoryReader,
)
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import ExactMatchFilter, MetadataFilters
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models as qmodels
from IPython.display import HTML, display

from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("tenant_rag")

# --- Config ---
TENANT_KEY = "tenant_id"          # metadata key used to scope every node to a tenant
COLLECTION_NAME = "tenant_documents"
QDRANT_PATH = "./qdrant_data"     # on-disk persistence, no server needed
ENABLE_HYBRID_SEARCH = True       # dense + sparse retrieval; see caveat in section 15
SIMILARITY_TOP_K = 3
CHUNK_SIZE = 512
CHUNK_OVERLAP = 100
MAX_RETRIES = 5


class TenantIsolationError(RuntimeError):
    '''Raised when an operation would violate tenant isolation.'''


def validate_tenant_id(tenant_id: str) -> None:
    if not tenant_id or not isinstance(tenant_id, str) or not tenant_id.strip():
        raise TenantIsolationError(
            "tenant_id must be a non-empty string. Refusing to ingest, "
            "query, or delete without one -- an empty tenant_id could "
            "match unintended data."
        )


# --- Retry/backoff for Gemini API calls ---
# Replaces a fixed `time.sleep(100)` guess with real exponential backoff
# that only retries on rate-limit / transient errors, and fails fast on
# things that will never succeed (bad key, malformed input).
_RETRYABLE_SIGNATURES = (
    "429", "resource_exhausted", "rate limit", "rate_limit", "quota",
    "timeout", "timed out", "503", "unavailable", "connection reset",
)

def _is_retryable(exc: BaseException) -> bool:
    text = f"{type(exc).__name__} {exc}".lower()
    return any(sig in text for sig in _RETRYABLE_SIGNATURES)

retrying = retry(
    reraise=True,
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=2, max=60),
    retry=retry_if_exception(_is_retryable),
)

## 4. Configure LLM + embedding models

The LLM stays on Gemini (needed for generation quality and tool use), but
embeddings now run locally via a Hugging Face sentence-transformers model
(`BAAI/bge-small-en-v1.5`) instead of calling the Gemini embedding API.

Trade-offs versus the Gemini embedding API worth knowing:
- **No API quota/cost for embeddings** — ingestion won't hit embedding
  rate limits at all, only the LLM calls can.
- **Runs on CPU in Colab** by default; slower per-document than a hosted
  API for very large corpora, but perfectly fine at this notebook's scale.
- **Downloads the model from Hugging Face on first use** — same kind of
  one-time network dependency as the `fastembed` BM25 model used for
  hybrid search below. If you're somewhere without access to
  `huggingface.co`, this will fail; pick a different available model or
  pre-cache it.
- **Fixed embedding dimension** (384 for `bge-small-en-v1.5`) — if you
  later re-ingest into an existing Qdrant collection with a different
  embedding model, dimensions must match or you'll need a fresh
  collection.

In [4]:
llm = GoogleGenAI(model="gemini-3.5-flash", api_key=api_key)

# Local Hugging Face embedding model -- no API key or Gemini quota used
# for embeddings. Swap model_name for a larger/smaller model as needed
# (e.g. "BAAI/bge-base-en-v1.5" for higher quality, "BAAI/bge-small-en-v1.5"
# for speed).
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.llm = llm
Settings.embed_model = embed_model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 5. Persistent multi-tenant vector store (Qdrant)

Replaces the notebook's original in-memory `VectorStoreIndex` (which loses
all data when the runtime resets) with Qdrant. `QDRANT_PATH` persists to
disk in this Colab session; set `QDRANT_URL` (and optionally
`QDRANT_API_KEY`) instead to point at a real Qdrant server or Qdrant Cloud
instance — same code either way.

We also create a payload index on `tenant_id` so tenant-filtered queries
and deletes are indexed lookups rather than full scans once you're on a
real Qdrant server (this has no effect in local/on-disk mode, which is
fine — it's just not needed at this scale).

In [5]:
import os

qdrant_url = os.environ.get("QDRANT_URL")
qdrant_api_key = os.environ.get("QDRANT_API_KEY")

if qdrant_url:
    qdrant_client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
else:
    Path(QDRANT_PATH).mkdir(parents=True, exist_ok=True)
    qdrant_client = QdrantClient(path=QDRANT_PATH)

vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    enable_hybrid=ENABLE_HYBRID_SEARCH,
    fastembed_sparse_model="Qdrant/bm25" if ENABLE_HYBRID_SEARCH else None,
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store, storage_context=storage_context)

try:
    qdrant_client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=TENANT_KEY,
        field_schema=qmodels.PayloadSchemaType.KEYWORD,
    )
except Exception as exc:
    logger.debug("Skipping payload index creation (expected in local mode): %s", exc)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

## 6. Download Data

We use three papers this time to demonstrate the system generalizes past
two hardcoded users: `An LLM Compiler for Parallel Function Calling`,
`Dense X Retrieval: What Retrieval Granularity Should We Use?`, and
`Chain-of-Thought Prompting Elicits Reasoning in Large Language Models`.

In [6]:
!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2312.04511.pdf" -O "llm_compiler.pdf"
!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2312.06648.pdf" -O "dense_x_retrieval.pdf"
!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2201.11903.pdf" -O "chain_of_thought.pdf"

--2026-08-09 18:48:48--  https://arxiv.org/pdf/2312.04511.pdf
Resolving arxiv.org (arxiv.org)... 151.101.131.42, 151.101.3.42, 151.101.67.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.131.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /pdf/2312.04511 [following]
--2026-08-09 18:48:48--  https://arxiv.org/pdf/2312.04511
Reusing existing connection to arxiv.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1020527 (997K) [application/pdf]
Saving to: ‘llm_compiler.pdf’

llm_compiler.pdf    100%[===================>] 996.61K  --.-KB/s    in 0.07s   

2026-08-09 18:48:48 (14.6 MB/s) - ‘llm_compiler.pdf’ saved [1020527/1020527]

--2026-08-09 18:48:48--  https://arxiv.org/pdf/2312.06648.pdf
Resolving arxiv.org (arxiv.org)... 151.101.131.42, 151.101.3.42, 151.101.67.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.131.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /pdf/2312.0664

## 7. Load Data

In [7]:
documents_jerry = SimpleDirectoryReader(input_files=["/content/dense_x_retrieval.pdf"]).load_data()
documents_ravi = SimpleDirectoryReader(input_files=["/content/llm_compiler.pdf"]).load_data()
documents_meera = SimpleDirectoryReader(input_files=["/content/chain_of_thought.pdf"]).load_data()

for name, docs in [("Jerry", documents_jerry), ("Ravi", documents_ravi), ("Meera", documents_meera)]:
    if not docs:
        raise ValueError(f"No documents were loaded for {name} -- check the download step above.")
    print(f"{name}: loaded {len(docs)} document(s)")

Jerry: loaded 19 document(s)
Ravi: loaded 22 document(s)
Meera: loaded 43 document(s)


## 8. Ingestion Pipeline

In [8]:
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP),
    ]
)

## 9. Hardened ingestion function

Unlike the original notebook (which just tagged `document.metadata["user"]`
and called `index.insert_nodes` directly), this function:

- Rejects an empty/missing `tenant_id` before touching anything.
- Excludes `tenant_id` from what's sent to the embedding model and the LLM,
  so it can't leak into embeddings or generated answers.
- Verifies every node produced by the pipeline is actually tagged with the
  correct tenant before it's allowed anywhere near the shared index — if
  even one node is untagged, the whole insert is aborted rather than
  silently letting untagged content into the index.
- Wraps both the pipeline run and the index insert in retry/backoff.

In [9]:
def ingest_documents(tenant_id: str, documents: list) -> int:
    '''Tag, chunk, validate, and index documents for a single tenant.
    Returns the number of nodes indexed.'''
    validate_tenant_id(tenant_id)
    if not documents:
        raise ValueError(f"No documents provided for tenant '{tenant_id}'.")

    for document in documents:
        document.metadata[TENANT_KEY] = tenant_id
        document.excluded_embed_metadata_keys = list(
            set(document.excluded_embed_metadata_keys or []) | {TENANT_KEY}
        )
        document.excluded_llm_metadata_keys = list(
            set(document.excluded_llm_metadata_keys or []) | {TENANT_KEY}
        )

    run_pipeline = retrying(pipeline.run)
    nodes = run_pipeline(documents=documents)

    if not nodes:
        raise RuntimeError(f"Ingestion pipeline produced zero nodes for tenant '{tenant_id}'.")

    untagged = [n for n in nodes if n.metadata.get(TENANT_KEY) != tenant_id]
    if untagged:
        raise TenantIsolationError(
            f"{len(untagged)} node(s) were missing the correct '{TENANT_KEY}' tag; "
            f"aborting insert to avoid leaking untagged content into the shared index."
        )

    insert_nodes = retrying(index.insert_nodes)
    insert_nodes(nodes)

    logger.info("Indexed %d node(s) for tenant '%s'", len(nodes), tenant_id)
    return len(nodes)

## 10. Ingest documents for each tenant

Retry/backoff on the ingestion calls themselves means we don't need a
blind `time.sleep(100)` between tenants — if a call hits a rate limit, it
backs off and retries automatically. A short pause between *tenants* is
still kept as a courtesy on free-tier quotas, but it's small and it's not
the only thing standing between you and a failed run.

In [10]:
ingest_documents("jerry", documents_jerry)
# time.sleep(5)
ingest_documents("ravi", documents_ravi)
# time.sleep(5)
ingest_documents("meera", documents_meera)
print("All tenants indexed successfully.")

All tenants indexed successfully.


## 11. Hardened query engine factory

This is the key structural fix versus the original notebook. There,
`jerry_query_engine` and `ravi_query_engine` were each built correctly by
hand, but nothing stopped a *future* `index.as_query_engine()` call
somewhere else in the notebook from being built without a filter and
silently returning every tenant's data.

`get_query_engine()` has no default for `tenant_id` and validates it —
there is no way to call this function and get back an engine that isn't
scoped to exactly one tenant.

In [11]:
def get_query_engine(tenant_id: str, similarity_top_k: int = SIMILARITY_TOP_K):
    validate_tenant_id(tenant_id)
    filters = MetadataFilters(filters=[ExactMatchFilter(key=TENANT_KEY, value=tenant_id)])
    kwargs = {"vector_store_query_mode": "hybrid"} if ENABLE_HYBRID_SEARCH else {}
    return index.as_query_engine(filters=filters, similarity_top_k=similarity_top_k, **kwargs)


def query_tenant(tenant_id: str, question: str) -> str:
    engine = get_query_engine(tenant_id)
    run_query = retrying(engine.query)
    return str(run_query(question))

## 12. Querying

In [12]:
import nest_asyncio
nest_asyncio.apply()

# Jerry has Dense X Retrieval and should be able to answer this.
response = query_tenant("jerry", "what are propositions mentioned in the paper?")
display(HTML(f'<p style="font-size:20px">{response}</p>'))

In [13]:
# Ravi has LLMCompiler and should be able to answer this.
response = query_tenant("ravi", "what are the steps involved in LLMCompiler?")
display(HTML(f'<p style="font-size:20px">{response}</p>'))

In [14]:
# Meera has Chain-of-Thought prompting and should be able to answer this.
response = query_tenant("meera", "what is chain-of-thought prompting?")
display(HTML(f'<p style="font-size:20px">{response}</p>'))

In [15]:
# This should NOT be answerable -- Jerry has no LLMCompiler content in his tenant scope.
response = query_tenant("jerry", "what are the steps involved in LLMCompiler?")
display(HTML(f'<p style="font-size:20px">{response}</p>'))

## 13. Isolation verification (inline tests)

Rather than just asserting isolation works by inspection, this cell
directly checks it: it retrieves raw nodes (bypassing the LLM) for each
tenant and asserts that **every single node returned is tagged with that
tenant, and none of the other tenants' distinctive content appears.** If
isolation were ever broken by a code change above, this cell would fail
loudly instead of the bug being discovered later via a data leak.

In [16]:
tenant_queries = {
    "jerry": ("propositions", ["LLMCompiler", "chain-of-thought", "chain of thought"]),
    "ravi": ("task", ["proposition", "chain-of-thought", "chain of thought"]),
    "meera": ("reasoning", ["proposition", "LLMCompiler"]),
}

for tenant_id, (search_term, forbidden_terms) in tenant_queries.items():
    retriever = get_query_engine(tenant_id).retriever
    nodes = retriever.retrieve(search_term)
    assert len(nodes) > 0, f"Expected results for tenant '{tenant_id}'"
    for n in nodes:
        actual_tenant = n.node.metadata.get(TENANT_KEY)
        assert actual_tenant == tenant_id, (
            f"ISOLATION VIOLATION: query for tenant '{tenant_id}' returned "
            f"a node tagged '{actual_tenant}'"
        )
        content = n.node.get_content()
        for forbidden in forbidden_terms:
            assert forbidden.lower() not in content.lower(), (
                f"ISOLATION VIOLATION: tenant '{tenant_id}' node contains "
                f"content mentioning '{forbidden}'"
            )
    print(f"PASSED: tenant '{tenant_id}' -- {len(nodes)} node(s), all correctly scoped")

print("\\nAll isolation checks passed.")

PASSED: tenant 'jerry' -- 3 node(s), all correctly scoped
PASSED: tenant 'ravi' -- 3 node(s), all correctly scoped
PASSED: tenant 'meera' -- 3 node(s), all correctly scoped
\nAll isolation checks passed.


## 14. Tenant deletion / lifecycle management

The original notebook had no way to remove a tenant's data. `delete_tenant`
purges every vector belonging to that tenant by filter, in one call.

In [17]:
def list_tenants(scan_limit: int = 10_000) -> set:
    '''Best-effort enumeration of distinct tenant_ids currently stored.
    Fine for admin/debug use; not intended for a hot path on very large
    collections -- for that, track tenant IDs in a separate small table.'''
    tenants, next_offset, scanned = set(), None, 0
    while scanned < scan_limit:
        points, next_offset = qdrant_client.scroll(
            collection_name=COLLECTION_NAME,
            limit=min(256, scan_limit - scanned),
            offset=next_offset,
            with_payload=True,
            with_vectors=False,
        )
        for point in points:
            tid = (point.payload or {}).get(TENANT_KEY)
            if tid:
                tenants.add(tid)
        scanned += len(points)
        if next_offset is None or not points:
            break
    return tenants


def delete_tenant(tenant_id: str) -> None:
    '''Permanently remove every vector belonging to a tenant.'''
    validate_tenant_id(tenant_id)
    condition = qmodels.FieldCondition(key=TENANT_KEY, match=qmodels.MatchValue(value=tenant_id))
    result = qdrant_client.delete(
        collection_name=COLLECTION_NAME,
        points_selector=qmodels.FilterSelector(filter=qmodels.Filter(must=[condition])),
        wait=True,
    )
    logger.info("Deleted tenant '%s' (status=%s)", tenant_id, result.status)

In [18]:
print("Tenants currently indexed:", list_tenants())

delete_tenant("meera")
print("Tenants after deleting 'meera':", list_tenants())

# Confirm Meera's content is actually gone, not just hidden.
remaining_nodes = get_query_engine("jerry").retriever.retrieve("reasoning")
assert not any(n.node.metadata.get(TENANT_KEY) == "meera" for n in remaining_nodes)
print("Confirmed: no 'meera' content remains in the index.")

Tenants currently indexed: {'meera', 'ravi', 'jerry'}
Tenants after deleting 'meera': {'ravi', 'jerry'}
Confirmed: no 'meera' content remains in the index.


## 15. Inspecting stored chunks in Qdrant

Useful for debugging: what does a chunk actually look like once it's
stored? This scrolls raw points straight out of the Qdrant collection —
bypassing LlamaIndex entirely — so you can see exactly what's on disk:
the chunk text, its tenant tag, and (if hybrid search is on) both the
dense and sparse vectors attached to it.

In [22]:
import json

def inspect_qdrant_chunks(limit=5, tenant_id=None):

    # Optional tenant filter
    scroll_filter = None

    if tenant_id:
        scroll_filter = qmodels.Filter(
            must=[
                qmodels.FieldCondition(
                    key="tenant_id",
                    match=qmodels.MatchValue(value=tenant_id)
                )
            ]
        )

    points, _ = qdrant_client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=scroll_filter,
        limit=limit,
        with_payload=True,
        with_vectors=False,   # We only want chunk + metadata
    )

    for i, point in enumerate(points):

        print("=" * 80)
        print(f"POINT {i + 1}")
        print("=" * 80)

        print("Point ID:")
        print(point.id)

        payload = point.payload or {}

        print("\n--- RAW PAYLOAD ---")
        print(json.dumps(payload, indent=2, default=str))

        # LlamaIndex stores the TextNode inside _node_content
        if "_node_content" in payload:

            node = json.loads(payload["_node_content"])

            print("\n--- CHUNK TEXT ---")
            print(node.get("text", ""))

            print("\n--- NODE METADATA ---")
            print(json.dumps(
                node.get("metadata", {}),
                indent=2,
                default=str
            ))

        print()


# See 5 chunks from the collection
inspect_qdrant_chunks(limit=5)

POINT 1
Point ID:
0048c8b0-23a5-46c7-8267-ab5daa12f9d6

--- RAW PAYLOAD ---
{
  "page_label": "8",
  "file_name": "dense_x_retrieval.pdf",
  "file_path": "/content/dense_x_retrieval.pdf",
  "file_type": "application/pdf",
  "file_size": 935698,
  "creation_date": "2026-08-09",
  "last_modified_date": "2024-10-07",
  "tenant_id": "jerry",
  "_node_content": "{\"id_\": \"0048c8b0-23a5-46c7-8267-ab5daa12f9d6\", \"embedding\": null, \"metadata\": {\"page_label\": \"8\", \"file_name\": \"dense_x_retrieval.pdf\", \"file_path\": \"/content/dense_x_retrieval.pdf\", \"file_type\": \"application/pdf\", \"file_size\": 935698, \"creation_date\": \"2026-08-09\", \"last_modified_date\": \"2024-10-07\", \"tenant_id\": \"jerry\"}, \"excluded_embed_metadata_keys\": [\"last_accessed_date\", \"tenant_id\", \"file_type\", \"file_size\", \"file_name\", \"creation_date\", \"last_modified_date\"], \"excluded_llm_metadata_keys\": [\"last_accessed_date\", \"tenant_id\", \"file_type\", \"file_size\", \"file_nam

## 16. Production notes & caveats

These weren't built into this notebook, and are flagged rather than
silently skipped:

- **No authentication layer.** `tenant_id` here is whatever gets passed to
  these functions. In a real deployed service, `tenant_id` must come from
  a verified session/JWT claim — never from an unvalidated request
  parameter — or one tenant could pass another tenant's ID and read their
  data.
- **`list_tenants()` is a scan.** Fine for admin/debug use here; for a
  large production collection, track tenant IDs in a separate small
  table/index rather than deriving them by scrolling Qdrant payloads.
- **Hybrid search downloads a model on first use.** `fastembed`'s BM25
  sparse encoder pulls its model files from Hugging Face the first time
  it's used. If you're running somewhere without outbound access to
  `huggingface.co`, set `ENABLE_HYBRID_SEARCH = False` above to fall back
  to dense-only search.
- **The dense embedding model also downloads from Hugging Face** on first
  use (`BAAI/bge-small-en-v1.5`, ~130MB). Same network dependency as
  hybrid search above — this one isn't optional, since it's what generates
  every embedding, but it only downloads once per runtime.
- **`qdrant-client` version pin.** See the note in Section 1 — re-check
  upstream compatibility before changing the pinned version.
- **On-disk persistence is local to this Colab runtime.** `QDRANT_PATH`
  persists across cells in this session but not across a fresh Colab
  runtime restart. For real persistence, set `QDRANT_URL` to point at a
  Qdrant server or Qdrant Cloud instance.